## 1. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install persim gudhi -q

import os, glob, gc, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict
from typing import Dict, List
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone
from persim import PersistenceImager
import persim.images_weights as weights
from gudhi import RipsComplex
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['axes.unicode_minus'] = False
print('All imports loaded.')

## 2. Ground Truth & 경로 설정

In [ ]:
M1=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3]]
M2=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,4],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,4,4],[2,2,3,3,3,3,3,3]]
M3=[[6,6,7,7,7,7,7,7],[6,6,6,7,7,7,7,7],[9,6,3,3,3,3,3,3],[9,10,3,4,4,3,3,4],[9,10,3,3,4,4,3,4],[9,10,3,4,4,4,4,4],[9,10,3,4,3,4,4,4],[9,10,3,4,3,4,4,4]]
M4=[[6,6,12,12,7,7,7,7],[6,6,12,12,7,7,7,7],[9,6,6,11,7,7,4,4],[9,9,6,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,4,4,4,4,4],[9,9,10,4,4,4,4,4]]
M5=[[6,6,12,12,12,12,7,7],[6,6,12,12,12,12,12,7],[9,9,6,11,11,11,12,11],[9,9,6,11,11,11,4,4],[9,9,13,13,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,10,10,4,4,4,4]]
M6=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,11],[9,9,6,11,11,11,11,11],[9,9,6,6,6,13,4,4],[9,9,6,13,13,4,4,4],[9,9,6,13,4,4,4,4],[9,9,6,13,4,4,4,4]]
M7=[[6,6,12,12,12,12,12,12],[9,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,12],[9,6,6,11,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,11,11,11,4],[9,9,6,6,13,13,4,4],[9,9,6,13,13,4,4,4]]
M8=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,6,11,11,11,11],[9,6,6,6,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,6,11,11,11],[9,9,6,6,13,13,11,11],[9,9,6,6,13,13,11,4]]
GROUND_TRUTH_M = np.asarray([M1,M2,M3,M4,M5,M6,M7,M8])

def get_label_from_index(task_id):
    idx = task_id - 1
    return GROUND_TRUTH_M[(idx%64)//8][idx//64][idx%8]

def extract_adjacent_phases(matrices):
    adj = set()
    for M in matrices:
        M = np.array(M)
        r,c = M.shape
        for i in range(r):
            for j in range(c):
                cur = M[i,j]
                for di,dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                    ni,nj = i+di,j+dj
                    if 0<=ni<r and 0<=nj<c and cur!=M[ni,nj]:
                        adj.add(tuple(sorted([int(cur),int(M[ni,nj])])))
    d = {}
    for p1,p2 in adj:
        d.setdefault(p1,[]).append(p2)
        d.setdefault(p2,[]).append(p1)
    return d

ADJACENT_PHASES = extract_adjacent_phases(GROUND_TRUTH_M)

def soft_accuracy_score(y_true, y_pred, adj=ADJACENT_PHASES):
    n = len(y_true)
    correct = sum(1 for t,p in zip(y_true,y_pred)
                  if t==p or (t in adj and p in adj[t]) or (p in adj and t in adj[p]))
    return correct/n if n>0 else 0.0

A_VALS = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAM_LIST = [(x1,x2,x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]

BASE_DIR = '/content/drive/MyDrive/URP'
VECTOR_DIR = os.path.join(BASE_DIR, '1224_Vectors')
BARCODE_CACHE = os.path.join(BASE_DIR, 'Phase3_Mixup_Barcodes')
os.makedirs(BARCODE_CACHE, exist_ok=True)
print(f'{len(PARAM_LIST)} parameter sets, ADJACENT_PHASES ready.')

## 3. Mixup Barcode 계산 함수
원본 `Mixup barcode.ipynb`에서 가져온 정확한 구현

In [ ]:
def compute_Rips(points, max_edge=10, max_dim=2):
    rips = RipsComplex(points=points, max_edge_length=max_edge)
    st = rips.create_simplex_tree(max_dimension=max_dim)
    return st

def extract_filtration(st):
    pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    simplices = [p[0] for p in pairs]
    filt_values = [p[1] for p in pairs]
    return simplices, filt_values

def _reduce_matrix(columns, n):
    R = [set(col) for col in columns]
    low = [-1] * n
    pivot_to_col = {}
    for j in range(n):
        while R[j]:
            pivot = max(R[j])
            if pivot in pivot_to_col:
                R[j] ^= R[pivot_to_col[pivot]]
            else:
                pivot_to_col[pivot] = j
                low[j] = pivot
                break
        else:
            low[j] = -1
    return R, low

def compute_mixup_barcode(A, B, max_edge=10, max_dim=1):
    """Compute mixup barcode (Algorithm 2, Wagner et al. 2024).
    Returns: Dict[int, ndarray of shape (N,3)] with columns [birth, image_death, domain_death]"""
    total = np.concatenate([A, B], axis=0)
    a = len(A)
    st = compute_Rips(total, max_edge, max_dim=max_dim+1)
    simplices, filt = extract_filtration(st)
    n = len(simplices)
    simplex_dims = [len(s) - 1 for s in simplices]
    sf_to_idx = {s: i for i, s in enumerate(simplices)}
    in_L = [all(v < a for v in s) for s in simplices]
    idx_L = [i for i, b in enumerate(in_L) if b]
    idx_KmL = [i for i, b in enumerate(in_L) if not b]
    row_order = idx_L + idx_KmL
    old_to_new_row = {old: new for new, old in enumerate(row_order)}
    n_L = len(idx_L)
    BK_original = []
    for s in simplices:
        if len(s) <= 1:
            BK_original.append(set())
        else:
            rows = set()
            for j in range(len(s)):
                face = s[:j] + s[j+1:]
                if face in sf_to_idx:
                    rows.add(sf_to_idx[face])
            BK_original.append(rows)
    BK = []
    for col_idx in range(n):
        new_rows = {old_to_new_row[r] for r in BK_original[col_idx]}
        BK.append(new_rows)
    BL = []
    for col_idx in range(n):
        if in_L[col_idx]:
            new_rows = {r for r in BK[col_idx] if r < n_L}
            BL.append(new_rows)
        else:
            BL.append(set())
    RL, lowL = _reduce_matrix(BL, n)
    RK, lowK = _reduce_matrix(BK, n)
    pivotL_to_col = {}
    for j in idx_L:
        if lowL[j] != -1:
            pivotL_to_col[lowL[j]] = j
    pivotK_to_col = {}
    for j in range(n):
        if lowK[j] != -1:
            pivotK_to_col[lowK[j]] = j
    mixup_triples = defaultdict(list)
    for sigma in idx_L:
        if RL[sigma]:
            continue
        dim = simplex_dims[sigma]
        if dim > max_dim:
            continue
        sigma_row = old_to_new_row[sigma]
        birth = filt[sigma]
        tau = pivotL_to_col.get(sigma_row, None)
        death = filt[tau] if tau is not None else np.inf
        tau_prime = pivotK_to_col.get(sigma_row, None)
        death_prime = filt[tau_prime] if tau_prime is not None else np.inf
        if not np.isinf(death) and (np.isinf(death_prime) or death_prime > death):
            death_prime = death
        if np.isinf(death) or abs(death - birth) > 1e-10:
            mixup_triples[dim].append((birth, death_prime, death))
    result = {}
    for dim in sorted(mixup_triples.keys()):
        triples = mixup_triples[dim]
        if triples:
            arr = np.array(triples)
            order = np.argsort(arr[:, 0])
            result[dim] = arr[order]
        else:
            result[dim] = np.empty((0, 3))
    for d in range(max_dim + 1):
        if d not in result:
            result[d] = np.empty((0, 3))
    return result

print('Mixup barcode functions defined.')

## 4. Inter_PI / 3D_PI 벡터화 함수 (Fixed)
**수정 사항:** `persim.PersistenceImager` 대신 `scipy.stats.norm.pdf` 직접 구현으로 `TypeError` 방지.

In [ ]:
def compute_Interaction_PIs(
    barcodes, max_eps=10, px_res=0.1, sigma=0.05,
    normalization=False,
    weight_type='mixup'  # 'mixup' | 'mixup_persistence'
):
    """Inter_PI 벡터화 (persim weight 미사용 - 직접 계산)."""
    vector = {}

    def get_weights(b, d_prime, d):
        if weight_type == 'mixup':
            return d - d_prime
        elif weight_type == 'mixup_persistence':
            return 2 * d - d_prime - b
        else:
            raise ValueError(f"Unknown weight_type: {weight_type}")

    def compute_pi_2d(bars_bd, w_arr, b_range, p_range, px, sig):
        """bars_bd: (N,2) with (birth, death), internally specific to Inter-PI range handling."""
        # Note: Inter_PI usually maps (birth, death) but we want specific ranges
        # Birth range usually (0, eps), Pers range (0, eps)
        # Create Grid
        n_b = max(1, int((b_range[1] - b_range[0]) / px))
        n_p = max(1, int((p_range[1] - p_range[0]) / px))
        b_grid = np.linspace(b_range[0] + px/2, b_range[1] - px/2, n_b)
        p_grid = np.linspace(p_range[0] + px/2, p_range[1] - px/2, n_p)
        
        img = np.zeros((n_b, n_p))
        if len(bars_bd) == 0: return img

        for k in range(len(bars_bd)):
            birth_k = bars_bd[k, 0]
            # For Inter_PI images, usually y-axis is persistence
            # But be careful with internal transform logic.
            # Here we assume standard PI logic: y = death - birth
            pers_k = bars_bd[k, 1] - bars_bd[k, 0]
            wk = w_arr[k]
            
            gb = norm.pdf(b_grid, loc=birth_k, scale=sig)
            gp = norm.pdf(p_grid, loc=pers_k, scale=sig)
            img += wk * np.outer(gb, gp)
        return img

    # H0
    bars_h0 = np.asarray(barcodes.get(0, np.zeros((0, 3))))
    if len(bars_h0) > 0:
        mask = np.isfinite(bars_h0).all(axis=1)
        bars_h0 = bars_h0[mask]
    if len(bars_h0) > 0:
        w0 = get_weights(bars_h0[:, 0], bars_h0[:, 1], bars_h0[:, 2])
        bd0 = np.stack([bars_h0[:, 0], bars_h0[:, 2]], axis=1)
        img_h0 = compute_pi_2d(bd0, w0, (0, 0.01), (0, max_eps), px_res, sigma)
    else:
        img_h0 = np.zeros((max(1, int(0.01/px_res)), int(max_eps/px_res)))
    vector[0] = np.mean(img_h0, axis=0)

    # H1
    bars_h1 = np.asarray(barcodes.get(1, np.zeros((0, 3))))
    if len(bars_h1) > 0:
        mask = np.isfinite(bars_h1).all(axis=1)
        bars_h1 = bars_h1[mask]
    if len(bars_h1) > 0:
        w1 = get_weights(bars_h1[:, 0], bars_h1[:, 1], bars_h1[:, 2])
        bd1 = np.stack([bars_h1[:, 0], bars_h1[:, 2]], axis=1)
        img_h1 = compute_pi_2d(bd1, w1, (0, max_eps), (0, max_eps/2), px_res, sigma)
    else:
        img_h1 = np.zeros((int(max_eps/px_res), int(max_eps/2/px_res)))
    vector[1] = img_h1.flatten()

    return vector

print('compute_Interaction_PIs defined (persim-free version).')

In [ ]:
def mixup_barcode_translation(mixup_barcode):
    translation = {}
    for i in range(2):
        translated = []
        for bar in mixup_barcode.get(i, []):
            if not (np.isfinite(bar[0]) and np.isfinite(bar[1]) and np.isfinite(bar[2])):
                continue
            translated.append([bar[0], bar[1] - bar[0], bar[2] - bar[0]])
        translation[i] = translated
    return translation

def compute_3d_persistence_image(
    mixup_barcodes, resolution=20,
    ranges=((0,10),(0,10),(0,10)), bandwidth=0.4,
    weight_type='ones'  # 'ones' | 'mixup' | 'mixup_persistence'
):
    """3D PI 벡터화. H0→2D(res^2), H1→3D(res^3).
    translated point = [b, d'-b, d-b]
    weight: ones=1, mixup=point[2]-point[1], mixup_persistence=2*point[2]-point[1]
    """
    translated = mixup_barcode_translation(mixup_barcodes)
    vectors = {}

    if weight_type == 'mixup':
        weight_func = lambda p: max(0, p[2] - p[1])
    elif weight_type == 'mixup_persistence':
        weight_func = lambda p: max(0, 2 * p[2] - p[1])
    else:
        weight_func = lambda p: 1.0

    x_grid = np.linspace(ranges[0][0], ranges[0][1], resolution)
    y_grid = np.linspace(ranges[1][0], ranges[1][1], resolution)
    z_grid = np.linspace(ranges[2][0], ranges[2][1], resolution)

    for i in range(2):
        barcode = translated[i]
        if i == 0:
            pi = np.zeros((resolution, resolution))
            for point in barcode:
                if not all(np.isfinite(v) for v in point): continue
                w = weight_func(point)
                gy = norm.pdf(y_grid, loc=point[1], scale=bandwidth)
                gz = norm.pdf(z_grid, loc=point[2], scale=bandwidth)
                pi += w * np.outer(gy, gz)
        else:
            pi = np.zeros((resolution, resolution, resolution))
            for point in barcode:
                if not all(np.isfinite(v) for v in point): continue
                w = weight_func(point)
                gx = norm.pdf(x_grid, loc=point[0], scale=bandwidth)
                gy = norm.pdf(y_grid, loc=point[1], scale=bandwidth)
                gz = norm.pdf(z_grid, loc=point[2], scale=bandwidth)
                pi += w * np.einsum('i,j,k->ijk', gx, gy, gz)
        vectors[i] = pi.flatten()
    return vectors

print('compute_3d_persistence_image defined.')

## 5. 전체 Mixup Barcode 계산 (분할 저장)

In [ ]:
MAX_EDGE = 10
SAVE_INTERVAL = 10
CACHE_FILE = os.path.join(BARCODE_CACHE, 'all_mixup_barcodes.npz')

if os.path.exists(CACHE_FILE):
    print('기존 캐시 파일 로드 중...')
    cached = np.load(CACHE_FILE, allow_pickle=True)
    all_mixup_barcodes = cached['barcodes'].item()
    print(f'✓ {len(all_mixup_barcodes)}개 샘플 로드됨')
else:
    all_mixup_barcodes = {}
    print('캐시 파일 없음. 처음부터 시작합니다.')

remaining = [idx for idx in range(1, 513) if idx not in all_mixup_barcodes]
print(f'계산 필요: {len(remaining)}개 / 전체 512개')

if len(remaining) == 0:
    print('✓ 모든 mixup barcode가 이미 계산되어 있습니다!')
else:
    print(f'계산 시작... (남은 {len(remaining)}개)')
    t0 = time.time()
    new_count = 0
    for i, idx in enumerate(remaining):
        params = PARAM_LIST[idx - 1]
        folder = os.path.join(BASE_DIR, f'ParamSweep_{idx}_Output')
        pos_file = os.path.join(folder, f'Pos_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat')
        types_file = os.path.join(folder, f'Types_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat')
        if not os.path.exists(pos_file):
            print(f'  [{idx}] 파일 없음, skip')
            continue
        sample_t0 = time.time()
        types = np.loadtxt(types_file, dtype=int)
        positions = np.loadtxt(pos_file, delimiter=',')
        A = positions[types == 1]
        B = positions[types == 2]
        bc_AtoB = compute_mixup_barcode(A, B, max_edge=MAX_EDGE)
        bc_BtoA = compute_mixup_barcode(B, A, max_edge=MAX_EDGE)
        all_mixup_barcodes[idx] = {
            'A_to_B': bc_AtoB, 'B_to_A': bc_BtoA,
            'label': get_label_from_index(idx),
        }
        new_count += 1
        if new_count % SAVE_INTERVAL == 0:
            print(f'    💾 중간 저장 ({len(all_mixup_barcodes)}개)...')
            np.savez_compressed(CACHE_FILE, barcodes=all_mixup_barcodes)
            print(f'    ✓ 저장 완료')
    np.savez_compressed(CACHE_FILE, barcodes=all_mixup_barcodes)
    print(f'✓ 최종 저장 완료.')

## 6. 다양한 조건으로 벡터화

In [ ]:
def vectorize_all(all_barcodes, vectorize_fn, **kwargs):
    X_list, y_list = [], []
    # Ensure deterministic order
    for idx in sorted(all_barcodes.keys()):
        entry = all_barcodes[idx]
        vec_AtoB = vectorize_fn(entry['A_to_B'], **kwargs)
        vec_BtoA = vectorize_fn(entry['B_to_A'], **kwargs)
        feats = np.concatenate([
            vec_AtoB[0], vec_AtoB[1],
            vec_BtoA[0], vec_BtoA[1]
        ])
        X_list.append(feats)
        y_list.append(entry['label'])
    return np.nan_to_num(np.array(X_list), nan=0., posinf=0., neginf=0.), np.array(y_list)

EXPERIMENTS = [
    {'name': 'Inter_PI (w=mixup)', 'fn': compute_Interaction_PIs, 'kw': {'weight_type': 'mixup'}, 'type': 'Inter_PI'},
    {'name': 'Inter_PI (w=mixup+pers)', 'fn': compute_Interaction_PIs, 'kw': {'weight_type': 'mixup_persistence'}, 'type': 'Inter_PI'},
    {'name': '3D_PI (w=ones)', 'fn': compute_3d_persistence_image, 'kw': {'weight_type': 'ones'}, 'type': '3D_PI'},
    {'name': '3D_PI (w=mixup)', 'fn': compute_3d_persistence_image, 'kw': {'weight_type': 'mixup'}, 'type': '3D_PI'},
    {'name': '3D_PI (w=mixup+pers)', 'fn': compute_3d_persistence_image, 'kw': {'weight_type': 'mixup_persistence'}, 'type': '3D_PI'},
]
print(f'{len(EXPERIMENTS)}개 실험 조건 정의됨.')

## 7. 평가 함수

In [ ]:
PCA_DIM = 20
N_SPLITS = 5

def evaluate_dataset(X, y, pca_dim=PCA_DIM, n_splits=N_SPLITS):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    if pca_dim and Xs.shape[1] > pca_dim:
        Xs = PCA(n_components=pca_dim, random_state=42).fit_transform(Xs)
    clfs = {
        'SVM-Linear': SVC(kernel='linear', C=1., random_state=42),
        'SVM-RBF': SVC(kernel='rbf', C=1., gamma='scale', random_state=42),
        'KNN(k=3)': KNeighborsClassifier(3),
        'RF': RandomForestClassifier(100, random_state=42),
    }
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = {}
    for cn, ct in clfs.items():
        soft_accs, strict_accs, f1s = [], [], []
        for tri, tei in skf.split(Xs, y):
            c = clone(ct); c.fit(Xs[tri], y[tri]); yp = c.predict(Xs[tei])
            soft_accs.append(soft_accuracy_score(y[tei], yp))
            strict_accs.append(accuracy_score(y[tei], yp))
            f1s.append(f1_score(y[tei], yp, average='weighted', zero_division=0))
        results[cn] = {
            'soft_mean':np.mean(soft_accs)*100, 'soft_std':np.std(soft_accs)*100,
            'strict_mean':np.mean(strict_accs)*100, 'strict_std':np.std(strict_accs)*100,
            'f1_mean':np.mean(f1s)*100, 'f1_std':np.std(f1s)*100,
        }
    return results

print('Evaluation function defined.')

## 8. 전체 실험 실행

In [ ]:
all_results = []

for i, exp in enumerate(EXPERIMENTS):
    print(f'\n[{i+1}/{len(EXPERIMENTS)}] {exp["name"]}...')
    t0 = time.time()
    X, y = vectorize_all(all_mixup_barcodes, exp['fn'], **exp['kw'])
    print(f'  Shape: {X.shape}, 벡터화 {time.time()-t0:.1f}s')
    results = evaluate_dataset(X, y)
    for cn, r in results.items():
        print(f'  {cn}: Soft={r["soft_mean"]:.2f}±{r["soft_std"]:.2f} | Strict={r["strict_mean"]:.2f}±{r["strict_std"]:.2f} | F1={r["f1_mean"]:.2f}')
    all_results.append({'name': exp['name'], 'type': exp['type'], 'dim': X.shape[1], 'results': results})
    del X, y; gc.collect()

print(f'\n✓ 전체 {len(all_results)}개 실험 완료')

## 9. 결과 비교 및 시각화

In [ ]:
# Build results DataFrame
rows = []
for exp in all_results:
    for clf_name, metrics in exp['results'].items():
        rows.append({
            'Method': exp['name'], 'Type': exp['type'], 'Dim': exp['dim'],
            'Classifier': clf_name,
            'Soft Acc': metrics['soft_mean'], 'Soft Std': metrics['soft_std'],
            'Strict Acc': metrics['strict_mean'], 'Strict Std': metrics['strict_std'],
            'F1': metrics['f1_mean'],
        })
df = pd.DataFrame(rows)

# Print SVM-Linear results
print('='*100)
print('SVM-Linear 결과 (PCA 20D)')
print('='*100)
df_svm = df[df['Classifier']=='SVM-Linear'].copy()
df_svm = df_svm.sort_values('Soft Acc', ascending=False)
print(df_svm[['Method','Dim','Soft Acc','Soft Std','Strict Acc','Strict Std','F1']].to_string(index=False))

# 기존 결과와 비교
print('\n=== 기존 최종.ipynb 결과와 비교 ===')
BASELINE_SOFT = {'Inter_PI': 97.27, '3D_PI': 96.87}
BASELINE_STRICT = {'Inter_PI': 72.86, '3D_PI': 71.67}
for _, row in df_svm.iterrows():
    baseline_key = 'Inter_PI' if 'Inter' in row['Method'] else '3D_PI'
    delta_soft = row['Soft Acc'] - BASELINE_SOFT[baseline_key]
    delta_strict = row['Strict Acc'] - BASELINE_STRICT[baseline_key]
    print(f'  {row["Method"]:30s} Soft={row["Soft Acc"]:.2f}% (Δ={delta_soft:+.2f}) Strict={row["Strict Acc"]:.2f}% (Δ={delta_strict:+.2f})')

df.to_csv('/content/drive/MyDrive/URP/phase3_final_results.csv', index=False)


## 10. 3D_PI Resolution / Bandwidth 탐색 (파라미터 스윕)

In [ ]:
RESOLUTIONS = [10, 15, 20, 25]
BANDWIDTHS = [0.1, 0.2, 0.4, 0.8]
WEIGHT_TYPES_3D = ['ones', 'mixup', 'mixup_persistence']

sweep_results = []
total_runs = len(RESOLUTIONS) * len(BANDWIDTHS) * len(WEIGHT_TYPES_3D)
count = 0

print(f'Starting parameter sweep (Total {total_runs} configs)...')
for wt in WEIGHT_TYPES_3D:
    for res in RESOLUTIONS:
        for bw in BANDWIDTHS:
            count += 1
            t0 = time.time()
            X, y = vectorize_all(
                all_mixup_barcodes,
                compute_3d_persistence_image,
                resolution=res, bandwidth=bw, weight_type=wt
            )
            results = evaluate_dataset(X, y)
            r_linear = results['SVM-Linear']
            r_rbf = results['SVM-RBF']
            print(f'[{count}/{total_runs}] w={wt} res={res} bw={bw} -> Soft(Lin)={r_linear["soft_mean"]:.1f}% Soft(RBF)={r_rbf["soft_mean"]:.1f}% ({time.time()-t0:.1f}s)')

            sweep_results.append({
                'weight': wt, 'resolution': res, 'bandwidth': bw,
                'dim': X.shape[1],
                'soft_acc_lin': r_linear['soft_mean'], 'soft_acc_rbf': r_rbf['soft_mean'],
                'strict_acc_lin': r_linear['strict_mean'], 'strict_acc_rbf': r_rbf['strict_mean'],
                'f1_lin': r_linear['f1_mean'], 'f1_rbf': r_rbf['f1_mean'],
            })
            del X, y; gc.collect()

df_sweep = pd.DataFrame(sweep_results)
df_sweep.to_csv('/content/drive/MyDrive/URP/phase3_3d_pi_sweep_final.csv', index=False)
print('Sweep Done. CSV Saved.')